<a id="top"></a>
# PanSTARRS "20 queries" using MAST's TAP service: Multiple detections


******

## Overview

This notebook is part of a series demonstrating how to address a set of scientific questions using SQL-like Astronomical Data Query Language (ADQL) queries via a Virtual Observatory standard Table Access Protocol (TAP) service at MAST. 

This series aims to be an introduction to how complex queries can be executed using TAP (which might otherwise not be possible to specify with `astroquery`), and to be a resource for how to access MAST databases after the MAST CASJobs service is retired.

The presented queries are drawn from 20 queries for SDSS as presented by [Gray, Szalay, et al. (2002)](https://arxiv.org/abs/cs/0202014), adapted for the PanSTARRS PS1 database.

This notebook presents the subset of queries to **find & analyze objects with multiple detections**, by joining object-level and detection catalogs and applying filtering constraints.


## Learning Goals
By the end of this tutorial, you will:

- Understand how to identify and filter objects with multiple detections by joining tables using ADQL queries with TAP services.


****
### Table of Contents

1. [Introduction](#Introduction)
1. [Imports](#Imports)
1. [Connect to TAP service](#Connect-to-TAP-service)
1. [Q15: Provide a list of moving objects consistent with an asteroid.](#Q15)
1. [Additional Resources](#Additional-Resources)
1. [About This Notebook](#About-this-Notebook)



*******
## Introduction

Welcome! This notebook demonstrates how to answer example scientific questions by performing SQL-like Astronomical Data Query Language (ADQL) queries accessing the PanSTARRS catalogs at MAST through a Virual Observatory Table Access Protocol (TAP) service.

This tutorial will demonstrate how to answer scientific questions that rely on obtaining information regarding multiple detections, by joining object-level catalogs with the Detection catalog (together with relevant filtering).

These queries are taken from (or closely modeled on) the "20 queries for SDSS" presented by [Gray, Szalay, et al. (2002)](https://arxiv.org/abs/cs/0202014). Taken as a whole, this collection provides worked examples on how to leverage relational database capabilities to answer scientific questions, with the aim of providing concrete starting points and references for designing queries in other research applications, both for PanSTARRS and other large data volume missions (including Roman).

<div class="alert alert-block alert-info">
<b>Note:</b> ADQL/SQL comments follow "--". Comments are used throughout the queries to explain the purpose of specific clauses.
</div>

The workflow for this notebook consists of:
* [Main](#Main)
* [Connect to TAP service](#Connect-to-TAP-service)
* [Q15: Provide a list of moving objects consistent with an asteroid.](#Q15)
  * [Constructing the query](#Q15:-Constructing-the-query)
  * [Inspecting & visualizing the results](#Q15:-Inspecting-&-visualizing-the-results)
* [Additional Resources](#Additional-Resources)
* [About This Notebook](#About-this-Notebook)

*(Note: a PanSTARRS implementation of Query 14 is planned to be added to this notebook at a future date.)*

## Imports
This tutorial makes use of the following libraries: 
- [*numpy*](https://numpy.org/) for numerical calculations
- [*pyvo*](https://pyvo.readthedocs.io) for querying the MAST catalogs via TAP
- [*matplotlib.pyplot*](https://matplotlib.org/stable/api/pyplot_summary.html#module-matplotlib.pyplot) for plotting data
- *time*, *datetime* to determine query duration
- [*astropy.table Table*](https://docs.astropy.org/en/stable/table/index.html) to load comparison data table

In [ ]:
import numpy as np
import pyvo as vo
import matplotlib.pyplot as plt
import datetime
import time
from astropy.table import Table

--------
## Connect to TAP service

For all queries, we will be connecting to the Pan-STARRS (PS1) Data release 2 (DR2) catalog. Specifically, we will connect to the new [PS1 DR2 postgres-backed TAP service](https://mast.stsci.edu/vo-tap/api/v0.1/ps1_dr2/), which offers improved performance relative to the legacy database (by factors of 100 or greater, in many cases).

See the [PS1 documentation](link_to_migration_guide_here) for information about the tables available with this new TAP service.
    
<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
<b>FIX LINK TO MIGRATION GUIDE ABOVE</b>
</div>

In [ ]:
TAP_service = vo.dal.TAPService(
    "https://mast.stsci.edu/vo-tap/api/v0.1/ps1_dr2"
)

This service supports the following ADQL features, 
and will return up to 100,000 rows.

In [ ]:
TAP_service.describe()

<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
The schemas for the PS1 DR2 tables are available at: https://mastdev.stsci.edu/schema_browser/#/
    <p></p>
</div>

<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
<i><b>**IF POSSIBLE:**</b></i>
<i>Integrete the schema browser.</i>

<i>Or else provide a simple URL link, so users can still use the schema browser even if it isn't fully integrated with the jupyter notebook?</i>
</div>

*********

## Q15

Query 15 poses: 

> **Provide a list of moving objects consistent with an asteroid.**


### Q15: Constructing the query

In the PS1 survey strategy, a pair or quad of images is acquired in a single filter, separated by approximately 15 minutes --- allowing for the identification of fast-moving near-Earth objects. (See Section 3.2 of the PS1 survey paper, [Chambers et al. 2016](https://ui.adsabs.harvard.edu/abs/2016arXiv161205560C/abstract), for more details.)

While there is a special PS1 database associated with these fast-moving objects, it is not include as part of the MAST PS1 databases. However, the data in the MAST PS1 database can still be used to identify fast-moving objects. The PS1 observing strategy groups observations in the `z` and `y` bands into groups of 4 observations for each filter, with a total of 8 observations collected within a 22 minute interval.  If the object is moving extremely fast, its detections for the 8 observations will appear in the database as 8 separate objects, each with a single detection.  That happens when the spacing between individual object positions is more than 1 arcsec (corresponding to an object proper motion greater than about 20 arcsec/hour).  Such objects are difficult to identify via a database query because it is necessary to find groups of objects spread over a few arcseconds, requiring a self-match of the object table with itself (a slow process).

For somewhat slower moving objects, however, the observations in consecutive images overlap, producing database entries that have 2 or more measurements depending on how fast the object is moving.  A slower moving object can have up to 8 measurements (4 in z and 4 in y).  If an object has at least 3 detections, a proper motion is included in the PS1 DR2 `mean_object` or `stack_object` table ([see the documentation on the new TAP service](link_to_migration_guide); in the legacy database, these are contained in the ObjectThin table).  The proper motion in these cases is vastly higher than the values for stars: the fastest moving star moves about 10 arcsec per year, while asteroids and other solar system objects can easily be moving by 10 arcsec per hour, nearly 10,000 times faster than the fastest stars.  

<div class="alert alert-warning" style="color:red; background-color:#ffc5c5; border-color:red;">
Add correct link to the migration guide in the above paragraph.
</div>

The query below searches for objects that move by more than 0.1 arcsec in 22 minutes (enough motion to have a chance of being detected by the PS1 pipeline).  It has many constraints to reduce contamination of the sample by spurious catalog sources:

- Require at least 4 detections (slightly more conservative than 3), with all detections in the `z` or `y` bands and zero detections in the `g`, `r`, or `i` bands.
- Require the range of times for all detections to be less than 1 hour.
- Require the proper motion to be greater than 0.1 arcsec of motion in 22 min
- Include only detection measurements of good quality: `psfqfperfect`>0.9 and `psfflux`>0 
- Exclude heavily resolved objects using the `kronflux`/`psfflux` ratio (because galaxies can have lower accuracy positions that mimic motion).
- Retain only observations with reasonably good resolution (psf FWHM <= 1.5 arcsec, for both major and minor axes).
- Require at least 4 remaining detections after the above constraints are applied.

This query is also limited to objects in slice 16 (by requiring `objid` between 
129500000000000000 and 133500000000000000), to ensure the query does not exceed the time limit for a single TAP query.

In [ ]:
# Proper motion threshold for 0.1 arcsec motion in 22 min, converted to milliarcsec per year
thresh = (100*(365.25e0*24*3600)/(22*60))  

# Construct query
adql_query = f"""
select ot.*
from (
    -- first select the objects having high PMs and >=4 z and y detections
    select 
        objid, ramean, decmean, rameanerr, decmeanerr,
        pmra, pmdec, pmraerr, pmdecerr, epochmean, posmeanchisq,
        ndetections, ng, nr, ni, nz, ny
    from mean_object
    where objid between 129500000000000000 and 133500000000000000             -- slice 16
      and ndetections>=4 and ng=0 and nr=0 and ni=0 and (nz+ny)=ndetections   -- only detections in y, z
      and sqrt(power(pmra,2)+power(pmdec,2)) > {thresh}                       -- proper motion threshold
) as ot
join (
    -- restrict to objects with all detections (high quality or not) within 1 hour time frame
    -- and objects with at least 4 detections after applying constraints/quality cuts.
    select d1.objid as objid, 
        d2.n_det as n_det
    from (
        -- select get observation time frame for all objects (no constraints/cuts)
        select objid, 
            max(obstime) as max_obstime, 
            min(obstime) as min_obstime
        from detection 
        group by objid
    ) as d1
    join (
        --- get count of detections meeting constraints/quality cuts
        select objid, count(objid) as n_det
        from detection
        where (psfqfperfect>0.9)
            and (psfflux>0)
            and (kronflux>0)
            and (-2.5*log10(psfflux)+8.90 < 20)
            and (2.5*log10(psfflux/kronflux) > -0.25)
            and (psfmajorfwhm <= 1.5)
            and (psfminorfwhm <= 1.5)
        group by objid
    ) as d2 on d1.objid=d2.objid
    where d2.n_det >= 4
        and (d1.max_obstime-d1.min_obstime)*86400 < 3600
) as d on d.objid=ot.objid
"""

In [ ]:
start = time.time()
job = TAP_service.run_async(adql_query, execution_duration=1200)
end = time.time()
print(f"Elapsed time: {str(datetime.timedelta(seconds=end-start))}")

### Q15: Inspecting & visualizing the results

This query takes about ~16 minutes to run and returned 316 rows. 

For all 32 slices, this would take a total time of ~8.5 hours and return ~10,000 rows.

In [ ]:
TAP_results = job.to_table()
TAP_results

The objects selected by this query have been tested by matching against the JPL Small Body Center catalog using the [Small Body Identification Tool](https://ssd.jpl.nasa.gov/tools/sb_ident.html). (Note that this tool is slow, with the search for the whole list taking approximately 12 hours, with multiple iterations to avoid timeout failures in some cases.)

33 of the 316 objects were confirmed as known objects. Information on these objects, including PS1 database proper motions and the JPL database proper motions --- both converted to the JPL units of arcsec/hour --- is available in the included file `q15-confirmed.ecsv`.

The plot below compares the PS1 database proper motions (y-axis) to the JPL database proper motions (x-axis). The RA and Dec proper motions are computed independently and are both shown on the plot.  There is excellent agreement between the PS1 PMs and the JPL PMs.  That provides support for this approach as a useful way to find moving objects.

In [ ]:
# Read in verified object aggregate table:
verified_objects = Table.read('q15-confirmed.ecsv')

# Plot PS1 versus JPL proper motions, which have been converted to units of arcsec/hour
f, ax = plt.subplots(figsize=(5.5, 5.5))
rmsarr = []
for key in ["RA", "Dec"]:
    ax.scatter(
        verified_objects[f"{key.lower()}pm"], # JPL/MPC values
        verified_objects[f"pm{key.lower()}"], # PS1 values
        label=key,
        s=15,
    )
    rmsarr.extend(
        verified_objects[f"pm{key.lower()}"]
        - verified_objects[f"{key.lower()}pm"]
    )
# Plot 1:1 line:
xlim = ax.get_xlim()
ylim = ax.get_ylim()
ax.plot(
    [np.min([xlim[0], ylim[0]]), np.max([xlim[1], ylim[1]])], 
    [np.min([xlim[0], ylim[0]]), np.max([xlim[1], ylim[1]])], 
    ls="--", color="tab:green", 
)
ax.set_xlim(xlim)
ax.set_ylim(ylim)

# Calculate RMS:
rms = np.sqrt(np.average(np.array(rmsarr)**2))
ax.legend(title=f"rms {rms:0.2f} arcsec/hr")
ax.set_xlabel("MPC Proper Motion [arcsec/hr]")
ax.set_ylabel("PS1 Proper Motion [arcsec/hr]")
_ = ax.set_title("33 objects with Minor Planet Center confirmation")

----------

## Additional Resources

### Table Access Protocol

- IVOA standard for RESTful web service access to tabular data
- http://www.ivoa.net/documents/TAP/

### PanSTARRS 1 DR 2

- https://outerspace.stsci.edu/display/PANSTARRS/

### Astronomical Query Data Language (2.0)

- IVOA standard for querying astronomical data in tabular format, with geometric search support
- http://www.ivoa.net/documents/latest/ADQL.html

### PyVO

- an affiliated package for [astropy](https://www.astropy.org/)
- find and retrieve astronomical data available from archives that support standard IVOA virtual observatory service protocols.
- https://pyvo.readthedocs.io/en/latest/index.html


### Full list of MAST/TAP services
- A full list of available MAST TAP services can be found at:
- https://mast.stsci.edu/vo-tap


## Citations
If you use `astropy` for published research, please cite the
authors. Follow these links for more information about citing `astropy`:

* [Citing `astropy`](https://www.astropy.org/acknowledging.html)

If you use PanSTARRS data accessed through MAST for published research, 
please include the following acknowledgements, found at the following links:

* [Acknowledging PanSTARRS](https://archive.stsci.edu/publishing/mission-acknowledgements#section-895d38a0-86b3-4143-b521-6cc3312701f9)
* [Acknowledging MAST](https://archive.stsci.edu/gsc/mast_data_use.html)


## About this Notebook


**Author(s):**  Rick White, Sedona Price<br>
**Keyword(s):** Tutorial, TAP, pyvo, ADQL, Pan-STARRS <br>
**Last Updated:** 2026-02-26 <br>
**Last Updated:** 2026-02-26
***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 